# 3️⃣ From Data to Decision


## 🏠 Can Young People Still Afford to Live in Brighton?

Welcome to the main hands-on project notebook.

In this notebook, you are a **Junior Data Analyst** investigating a real local question:

> **Can young people still afford to live in Brighton?**

By the end, you will have DBT models, DBT tests, charts, a simple decision checker, an optional Streamlit app, and a short stakeholder summary.

# 🔗 Public Data Sources Used

This notebook uses a prepared teaching dataset based on public source figures and locally relevant assumptions.

Source links:

- ONS Brighton & Hove housing page: https://www.ons.gov.uk/visualisations/housingpriceslocal/E06000043/
- ONS Private rent and house prices bulletin: https://www.ons.gov.uk/economy/inflationandpriceindices/bulletins/privaterentandhousepricesuk/march2026
- GOV.UK National Minimum Wage rates: https://www.gov.uk/national-minimum-wage-rates
- Brighton & Hove Buses fare updates: https://www.buses.co.uk/fares-2026
- Brighton & Hove Buses tickets page: https://www.buses.co.uk/tickets

Source benchmarks:

- ONS reports average monthly private rent in Brighton & Hove was **£1,826 in March 2026**.
- ONS reports average one-bedroom rent in Brighton & Hove was **£1,198 in March 2026**.
- GOV.UK lists April 2026 minimum wage rates as **£8.00 apprentice**, **£10.85 for ages 18–20**, and **£12.71 for ages 21+**.
- Brighton & Hove Buses lists 2026 fares including **£3 single tickets** and citySAVER day-ticket options.

For reliability, we do not live-download data during the workshop. We create small CSV files from source-backed assumptions.

In [ ]:
import sys
!{sys.executable} -m pip install duckdb pandas

# ❯❯❯❯ Part 1 — Execution Environment 



## ✅ Before You Start

Complete Notebook 2 first:

`02_environment/From_Zero_to_Working_DBT.ipynb`

That should have created `.venv/` and a `dbt_projects/p00_setup` with a working DBT folder structure used for validating your environment setup.

As we proceed, a similar folder structure will be built for the **Affordability Project**. 

Each project in this workshop will be fully contained in its own *project folder*. In our case the folder is 'dbt_projects/p01_affordability'. 

But before anything, we need to activate the proper Virtual Environment, **.venv**.


## ✋ Step 1.1 -  “Kernel Guard” - Activating your Virtual Environment  

Depending on the tool you are using to view your Jupyter Notebooks (e.g. Notebook, JupyterLab, VS Code), you need to activate the workshop Python environment (**.venv**).

How to do this:

- Look for a **Kernel** or a **Python Interpreter selector**. Usually, this can be found in the top or bottom right hand corner of your screen.
- Choose the interpreter inside the **.venv folder**
- If any code cell has been executed, restart the notebook and run all cells again

---

In [ ]:
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (
            (candidate / "README.md").exists()
            and (candidate / "requirements.txt").exists()
            and (candidate / "02_environment").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the workshop repo root.\n"
        "Open this notebook from inside the cloned dbt-workshop repository."
    )

repo_root = find_repo_root(Path.cwd())

expected_venv = repo_root / ".venv"
current_environment = Path(sys.prefix)

print("Current Python executable:")
print(sys.executable)

print("\nCurrent Python environment:")
print(current_environment)

print("\nExpected workshop environment:")
print(expected_venv)

if current_environment != expected_venv:
    print("\n⚠️ You are NOT using the workshop environment.")
    print("Please switch your notebook kernel/interpreter to:")
    print(expected_venv)
    print("\nThen restart the notebook and run all cells again.")
else:
    print("\n✅ Correct workshop environment is active")

### Verifing that the Pandas Library is available

Just ensuring that Pandas are available when we generate the seed files.

In [ ]:
try:
    import pandas as pd
    print("✅ Pandas is available:", pd.__version__)
except ImportError:
    print("❌ Pandas is NOT available in this environment")

## 🔍 Step 1.2 - Find the Repo and Tools

This cell finds the workshop repo, DBT projects folder, and virtual environment, as well as creating the profiles and database paths.


In [ ]:
from pathlib import Path
import os, platform, re, subprocess, sys, textwrap

#--------------------------------------------#
def find_repo_root(start: Path) -> Path:
  candidates = [start] + list(start.parents)

  for candidate in candidates:
      if (
          (candidate / "README.md").exists()
          and (candidate / "requirements.txt").exists()
          and (candidate / "02_environment").exists()
      ):
          return candidate

  raise FileNotFoundError(
      "Could not find the workshop repo root. "
      "Open this notebook from inside the cloned dbt-workshop repo."
  )

#--------------------------------------------#
#  Determine OS, folder names and file names 
#--------------------------------------------#
OS_NAME = platform.system()

repo_root = find_repo_root(Path.cwd())

venv_dir = repo_root / ".venv"

venv_python = venv_dir / (
  "Scripts/python.exe" if OS_NAME == "Windows" else "bin/python"
)

venv_pip = venv_dir / (
  "Scripts/pip.exe" if OS_NAME == "Windows" else "bin/pip"
)

venv_dbt = venv_dir / (
  "Scripts/dbt.exe" if OS_NAME == "Windows" else "bin/dbt"
)

requirements_file = repo_root / (
  "requirements_locked.txt"
  if (repo_root / "requirements_locked.txt").exists()
  else "requirements.txt"
)

project_name = "p01_affordability"

dbt_project_dir = repo_root / "dbt_projects" / project_name
dbt_project_dir.mkdir(parents=True, exist_ok=True)

profiles_dir = dbt_project_dir
database_path = (dbt_project_dir / "workshop.duckdb").resolve()

# project_data_dir = dbt_project_dir / "data"
# project_data_dir.mkdir(parents=True, exist_ok=True)

output_dir = dbt_project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

#-------------------------------------------------#
# Print out OS, Python Version, Repo Root Folder
# Project Folder and Installation Requirement File
#-------------------------------------------------#
print(f"Python: {sys.version.split()[0]}")
print(f"Operating System: {OS_NAME}")
print(sys.executable)

print(f"Repo root: {repo_root}")

print("DBT project exists:", dbt_project_dir.exists())
print("Virtual environment Python exists:", venv_python.exists())
print("Database:", database_path)
# print("Project data folder:", project_data_dir)
print("Output folder:", output_dir)



### ✅ Checkpoint

You should see:

- a **Python version**
- your **operating system**
- the path to your **repo root**
- **DBT project**, **database**, project data folder**, and **output folder**.
- a confirmation that your **project folder** and the **virtual environment** are ready to go
- the path to your **DuckDB database**
- the path to your **data folder**
- the path to your **output files folder**

If `dbt_projects/` does not exist, complete Notebook 2 first.

## Step 1.3 — Helper function

This helper runs terminal commands and shows the output inside the notebook.

---

In [ ]:
def run_command(cmd, cwd=None, check=True):
    print("Running:", " ".join(str(x) for x in cmd))
    completed = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True
    )

    if completed.stdout.strip():
        print(completed.stdout)

    if completed.stderr.strip():
        print(completed.stderr)

    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")

    return completed

### ✅ Checkpoint

The helper function is ready.

# ❯❯❯❯ Part 2 — Infrastructure and Tools 

Normally, many people start with `dbt init`, but that command is interactive.

For this workshop, we will build the project step by step so you can clearly see what each piece does.


## 🏗️ Step 2.1 — Build your DBT project folder structure

As we perform the Data Analysis, we deal with a wide variety of assets, either imported or generated. A clear project folder structure that a abides to conventional practice facilitates understanding the process of Data Analysis and to find issues if they arise.

The following code builds a "common practice" folder structure for our Data Analysis project.


In [ ]:
from pathlib import Path

seeds_dir = dbt_project_dir / "seeds"
models_dir = dbt_project_dir / "models"
tests_dir = dbt_project_dir / "tests"
analysis_dir = dbt_project_dir / "analysis"
macros_dir = dbt_project_dir / "macros"

output_dir = dbt_project_dir / "output"
charts_dir = output_dir / "charts"
reports_dir = output_dir / "reports"
app_dir = output_dir / "app"

for folder in [
        seeds_dir, 
        models_dir,
        tests_dir, 
        analysis_dir,
        macros_dir,
        charts_dir,
        reports_dir,
        app_dir
    ]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ DBT project folders created at:", dbt_project_dir)



### ✅ Checkpoint

You should now have a folder structure like:

```text
├── dbt_projects/
│   └── p01_affordability/
│       ├── models/
│       ├── seeds/
│       ├── tests/
│       ├── analysis/
│       ├── macros/
│       └── output/
│           ├── charts/
│           ├── reports/
│           └── app/
```

That means your DBT project structure now exists locally.

---

In [ ]:
print("Here is your DBT project structure so far:\n")

for item in sorted(dbt_project_dir.iterdir()):
    print("-", item.name)


## 📄 Step 2.2 — Create `dbt_project.yml`

This is the main DBT project configuration file.

It tells DBT:

- what the project is called
- where your models live
- where seeds live
- how models should be built


In [ ]:

import textwrap

dbt_project_yml = dbt_project_dir / "dbt_project.yml"

if dbt_project_yml.exists():
    print("✅ dbt_project.yml file already exists")
    
else:
    dbt_project_yml.write_text(
        textwrap.dedent(f"""\
        name: '{project_name}'
        version: '1.0'
        config-version: 2

        profile: '{project_name}'

        model-paths: ["models"]
        analysis-paths: ["analysis"]
        test-paths: ["tests"]
        seed-paths: ["seeds"]
        macro-paths: ["macros"]
        snapshot-paths: ["snapshots"]

        clean-targets:
          - "target"
          - "dbt_packages"

        models:
          {project_name}:
              +materialized: view

        """),
        encoding="utf-8"
    )

    print("✅ Created:", dbt_project_yml)
    
print()
print(dbt_project_yml.read_text(encoding="utf-8"))


### ✅ Checkpoint

You should now have a file:

`dbt_projects/p01_affordability/dbt_project.yml`

This file is the **heart** of your DBT project.



## 🔌 Step 2.3 — Create `profiles.yml`

DBT also needs a separate file that tells it **how to connect to a database**.

For this workshop, we use **DuckDB** because it is small, local, and easy to set up.


In [ ]:

profiles_yml = dbt_project_dir / "profiles.yml"

if profiles_yml.exists():
    print("✅ Profile.yml file already exists")
    
else:
    profiles_yml.write_text(
        f"""{project_name}:
      target: dev
      outputs:
        dev:
          type: duckdb
          path: "{database_path.as_posix()}"
          threads: 1
    """,
        encoding="utf-8"
    )

    print('✅ Profile created')
    
print()
print(f'Project name: {project_name}')
print(f'Profile file: {profiles_yml}')
print(f'Database file: {database_path}')

print()
print(profiles_yml.read_text(encoding="utf-8"))


### ✅ Checkpoint

You should see:
- `Profile created`
- your DBT project name
- a DuckDB file path name ending in `workshop.duckdb` written in the `profiles.yml` file

Also, you should now have:

`dbt_projects/p01_affordability/profiles.yml`

This file connects your DBT project to a local DuckDB database.


## 📊  Step 2.4 — Create the Project Datasets

We will create four small CSV files based upon source-backed assumptions.

- `affordability_rents.csv`
- `affordability_personas.csv`
- `affordability_living_costs.csv`
- `affordability_transport.csv`


In [ ]:
import sys
print(sys.executable)

import pandas as pd

rents = pd.DataFrame([
    {"area": "Brighton Centre", "rent_option": "shared_room", "monthly_rent": 775, "notes": "Workshop estimate for shared room"},
    {"area": "Brighton Centre", "rent_option": "one_bed", "monthly_rent": 1250, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "North Laine", "rent_option": "shared_room", "monthly_rent": 800, "notes": "Workshop estimate for shared room"},
    {"area": "North Laine", "rent_option": "one_bed", "monthly_rent": 1280, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Hove", "rent_option": "shared_room", "monthly_rent": 850, "notes": "Workshop estimate for shared room"},
    {"area": "Hove", "rent_option": "one_bed", "monthly_rent": 1300, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Kemptown", "rent_option": "shared_room", "monthly_rent": 760, "notes": "Workshop estimate for shared room"},
    {"area": "Kemptown", "rent_option": "one_bed", "monthly_rent": 1160, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Brighton Marina", "rent_option": "shared_room", "monthly_rent": 825, "notes": "Workshop estimate for shared room"},
    {"area": "Brighton Marina", "rent_option": "one_bed", "monthly_rent": 1200, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Moulsecoomb", "rent_option": "shared_room", "monthly_rent": 680, "notes": "Workshop estimate for shared room"},
    {"area": "Moulsecoomb", "rent_option": "one_bed", "monthly_rent": 1000, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
])

personas = pd.DataFrame([
    {"persona": "18-year-old apprentice", "age_group": "18-20", "employment_type": "Apprentice", "hourly_rate": 8.00, "hours_per_week": 37.5, "monthly_take_home_estimate": 1260, "notes": "Based on 2026 apprentice minimum wage; simplified monthly estimate"},
    {"persona": "19-year-old retail worker", "age_group": "18-20", "employment_type": "Retail / Hospitality", "hourly_rate": 10.85, "hours_per_week": 37.5, "monthly_take_home_estimate": 1660, "notes": "Based on 2026 18-20 minimum wage; simplified monthly estimate"},
    {"persona": "21-year-old full-time worker", "age_group": "21+", "employment_type": "Entry-level full-time", "hourly_rate": 12.71, "hours_per_week": 37.5, "monthly_take_home_estimate": 1900, "notes": "Based on 2026 National Living Wage; simplified monthly estimate"},
    {"persona": "23-year-old graduate analyst", "age_group": "21+", "employment_type": "Graduate role", "hourly_rate": 15.00, "hours_per_week": 37.5, "monthly_take_home_estimate": 2250, "notes": "Workshop scenario for early-career graduate income"},
])

living_costs = pd.DataFrame([
    {"cost_item": "food", "monthly_cost": 250, "category": "essential"},
    {"cost_item": "utilities_share", "monthly_cost": 120, "category": "essential"},
    {"cost_item": "mobile_phone", "monthly_cost": 25, "category": "essential"},
    {"cost_item": "internet_share", "monthly_cost": 25, "category": "essential"},
    {"cost_item": "basic_personal_spending", "monthly_cost": 150, "category": "essential"},
])

transport = pd.DataFrame([
    {"transport_option": "walk_or_cycle", "monthly_transport_cost": 20, "notes": "Low-cost active travel assumption"},
    {"transport_option": "bus_commuter", "monthly_transport_cost": 120, "notes": "Approx. two capped single fares per weekday"},
    {"transport_option": "daily_city_bus", "monthly_transport_cost": 136, "notes": "Approx. 20 adult citySAVER day tickets"},
])

seeds_dir = dbt_project_dir / "seeds"

rents.to_csv(seeds_dir / "affordability_rents.csv", index=False)
personas.to_csv(seeds_dir / "affordability_personas.csv", index=False)
living_costs.to_csv(seeds_dir / "affordability_living_costs.csv", index=False)
transport.to_csv(seeds_dir / "affordability_transport.csv", index=False)

print("✅ Created project CSV files:")
for path in sorted(seeds_dir.glob("affordability_*.csv")):
    print("-", path.name)
    

### ✅ Checkpoint

You should now have four CSV files:

- `dbt_projects/p01_affordability/seeds/affordability_rents.csv` 
- `dbt_projects/p01_affordability/seeds/affordability_personas.csv` 
- `dbt_projects/p01_affordability/seeds/affordability_living_costs.csv` 
- `dbt_projects/p01_affordability/seeds/affordability_transport.csv` 

Nice — you now have raw data ready to load.


## 👁️👁️ Step 2.5 — Explore the Raw Data

Before building transformations, analysts first inspect the raw datasets.

At this stage, the CSV files have been created inside the DBT `seeds/` folder, but they are still simple source data.

We will now take a quick look at the tables to understand:
- what information is available
- how the data is structured
- what each dataset represents

This is an important habit in real-world Data Analysis:
always explore the data before transforming it.

In [ ]:
rents = pd.read_csv(seeds_dir / "affordability_rents.csv")
personas = pd.read_csv(seeds_dir / "affordability_personas.csv")
living_costs = pd.read_csv(seeds_dir / "affordability_living_costs.csv")
transport = pd.read_csv(seeds_dir / "affordability_transport.csv")

print("✅ CSV files created in DBT seeds folder")

display(rents.head())
display(personas.head())
display(living_costs.head())
display(transport.head())


### ✅ Checkpoint

You should now see four small datasets representing different aspects of affordability in Brighton.

Each table contains only part of the overall picture:

- rents
- personas and income assumptions
- living costs
- transport costs

At this stage, the data is still fragmented and difficult to interpret directly.

In the next steps, DBT will help us:
- combine these datasets
- apply calculations and business logic
- create analytical models that can support decision-making

This is the transition from:  raw data → analytical insight


## 🧮 Step 2.6 — Create DBT Models

Now that we have explored the raw datasets, it’s time to create transformation models in DBT.

We will build two types of models:

1. **Staging model (`stg_affordability_inputs`)**  
   - Combines and standardizes the raw datasets  
   - Prepares the data for analysis  

2. **Decision-ready mart (`mart_affordability_scenarios`)**  
   - Aggregates inputs into actionable metrics  
   - Calculates affordability status for each persona and scenario

This step transforms raw CSV data into structured, analytical tables that can be queried and visualized.  

💡 Think of it as turning ingredients into a recipe: raw inputs are now combined into something useful.

In [ ]:
models_dir = dbt_project_dir / "models"

staging_dir = models_dir / "staging"
staging_dir.mkdir(parents=True, exist_ok=True)

marts_dir = models_dir / "marts"
marts_dir.mkdir(parents=True, exist_ok=True)

staging_model = staging_dir / "stg_affordability_inputs.sql"

if staging_model.exists():
    print("✅ stg_affordability_inputs.sql model file already exists")
    
else:
    staging_model.write_text(textwrap.dedent(
    """\
    with base_costs as (
        select sum(monthly_cost) as monthly_base_living_cost
        from {{ ref('affordability_living_costs') }}
    ),
    
    scenarios as (
        select
            p.persona,
            p.age_group,
            p.employment_type,
            p.monthly_take_home_estimate,
            r.area,
            r.rent_option,
            r.monthly_rent,
            t.transport_option,
            t.monthly_transport_cost,
            b.monthly_base_living_cost
        from {{ ref('affordability_personas') }} as p
        cross join {{ ref('affordability_rents') }} as r
        cross join {{ ref('affordability_transport') }} as t
        cross join base_costs as b
    )
    
    select * from scenarios
    """),
    encoding="utf-8"
    )


mart_model = marts_dir / "mart_affordability_scenarios.sql"

if mart_model.exists():
    print("✅ mart_affordability_scenarios.sql model file already exists")
    
else:
    mart_model.write_text(textwrap.dedent(
    """\
    with scenarios as (
        select * from {{ ref('stg_affordability_inputs') }}
    ),
    
    calculated as (
        select
            persona,
            age_group,
            employment_type,
            monthly_take_home_estimate,
            area,
            rent_option,
            monthly_rent,
            transport_option,
            monthly_transport_cost,
            monthly_base_living_cost,
            monthly_take_home_estimate - monthly_rent - monthly_transport_cost - monthly_base_living_cost as monthly_leftover,
            round((monthly_rent * 100.0) / monthly_take_home_estimate, 1) as rent_burden_pct
        from scenarios
    )
    
    select
        *,
        case
            when monthly_leftover >= 300 and rent_burden_pct <= 35 then 'Affordable'
            when monthly_leftover >= 0 then 'Tight'
            else 'Not affordable'
        end as affordability_status
    from calculated
    """),
    encoding="utf-8")
    

print("✅ Created DBT models:")
print()
print("-", staging_model)
print()
print(staging_model.read_text(encoding="utf-8"))
print("\n------------------------------------------------\n")
print("-", mart_model)
print()
print(mart_model.read_text(encoding="utf-8"))
                          

### ✅ Checkpoint — Models Created

You should now see:

- `stg_affordability_inputs.sql` in `models/staging/`  
  → your raw datasets are combined and ready for calculations  

- `mart_affordability_scenarios.sql` in `models/marts/`  
  → contains the affordability metrics and decision-ready outputs

What we did:

- Created the folder structure for `staging/` and `marts/`
- Generated SQL files for your transformations
- Prepared the foundation for the analytical workflow

Next steps:

- DBT will run these models to create the tables in the DuckDB database
- After running, you can inspect the outputs and start analyzing them

💡 Tip: Always check that your models produce the expected columns and calculations before moving on to dashboards or visualizations.

## 🛡️ Step 2.7 — Add Data Quality Checks
  
Before we trust data, we should test it.  
  
That’s one of the really useful things DBT does well ✅  
  
In this step, you will add rules to check, for example, that:  
  
- 💷 `rent_amount` is greater than zero
- 👥 `rent_option`, 'age_group" each are one of allowed values
  
Think of this as a **quality gate** for your data.  
  
If the data breaks the rules, DBT will tell you.

---


## 🧩 Step 2.7a — Create a Custom Test Rule
  
  DBT already includes some built-in tests like:  
  
- `not_null`
- `unique`
- `accepted_values`
  
  But for:  
  
  >   “amount spent must be greater than zero”  
  >   “amount spent must be non-negative”  
  
  we will create our own small reusable tests.

---

In [ ]:
macros_dir = dbt_project_dir / "macros"
macros_dir.mkdir(parents=True, exist_ok=True)

macro_file_pv = macros_dir / "test_positive_value.sql"

if macro_file_pv.exists():
    print("✅ Generic test positive value macro file already exists")
    
else:
    macro_file_pv.write_text(
        textwrap.dedent("""\
        {% test positive_value(model, column_name) %}
        
        select *
        from {{ model }}
        where {{ column_name }} <= 0
        
        {% endtest %}
        """), 
        encoding="utf-8"
    )

    print("✅ Generic test positive value macro created\n")


macro_file_nnv = macros_dir / "test_non_negative_value.sql"

if macro_file_nnv.exists():
    print("✅ Generic test non-negative value macro file already exists")
    
else:
    macro_file_nnv.write_text(
        textwrap.dedent("""\
        {% test non_negative_value(model, column_name) %}
        
        select *
        from {{ model }}
        where {{ column_name }} < 0
        
        {% endtest %}
        """), 
        encoding="utf-8"
    )

    print("✅ Generic test non-negative value macro created\n")

print("✅ Created custom DBT test macros")

### ✅ Checkpoint

You should see custom test macros created.

  You should see: 
  
- a success message
- a path ending in:
  
  ```
  dbt_projects/p01_affordability/macros/test_positive_value.sql
  ```
- ### 💡 What just happened?
  
  You created a reusable DBT test called:  
  
  ```
  positive_value
  ```
  
  It checks for rows where a numeric value is **zero or less**.  
  
  In our case, that means:  
  
  >   “show me any rows where `amount_spent` is not valid”

---

## 📝 Step 2.7b — Create  `setup_schema.yml` for applying Validation Rules 
  
  Now we define the actual rules DBT should apply to the data.  
  
  This file tells DBT:  
  
- which table to test
- which columns to check
- which rules must pass

---

In [ ]:
schema_file = models_dir / "schema.yml"

if schema_file.exists():
    print("✅ setup_schema.yml Validation Rules file already exists")
    
else:
    schema_file.write_text(
        textwrap.dedent("""\
        version: 2
        
        seeds:
          - name: affordability_rents
            columns:
              - name: area
                tests:
                  - not_null
              - name: rent_option
                tests:
                  - not_null
                  - accepted_values:
                      arguments:
                          values: ['shared_room', 'one_bed']
              - name: monthly_rent
                tests:
                  - not_null
                  - positive_value
        
          - name: affordability_personas
            columns:
              - name: persona
                tests:
                  - not_null
                  - unique
              - name: age_group
                tests:
                  - not_null
                  - accepted_values:
                      arguments:
                          values: ['18-20', '21+']
              - name: monthly_take_home_estimate
                tests:
                  - not_null
                  - positive_value
        
          - name: affordability_living_costs
            columns:
              - name: cost_item
                tests:
                  - not_null
                  - unique
              - name: monthly_cost
                tests:
                  - not_null
                  - non_negative_value
        
          - name: affordability_transport
            columns:
              - name: transport_option
                tests:
                  - not_null
                  - unique
              - name: monthly_transport_cost
                tests:
                  - not_null
                  - non_negative_value
        
        models:
          - name: stg_affordability_inputs
            description: "Combined input scenarios for Brighton affordability analysis."
          - name: mart_affordability_scenarios
            description: "Decision-ready affordability scenarios."
            columns:
              - name: persona
                tests:
                  - not_null
              - name: area
                tests:
                  - not_null
              - name: monthly_leftover
                tests:
                  - not_null
              - name: rent_burden_pct
                tests:
                  - not_null
              - name: affordability_status
                tests:
                  - not_null
                  - accepted_values:
                      arguments:
                          values: ['Affordable', 'Tight', 'Not affordable']
        """),
        encoding="utf-8"
    )
    print("✅ Created schema.yml")
    
print()
print(schema_file.read_text(encoding="utf-8"))

### ✅ Checkpoint
  
  You should see:  
  
- a success message
- the contents of `schema.yml` printed in the notebook

### 🔍 Rules you just added
  
For the seed table **'affordability_rents'**:  
- `area` must not be empty
- `rent_option' must be one of: 
	- `shared_room`
	- `one_bed`
- `monthly_rent` must be greater than zero  

  
For the seed table **'affordability_personas'**:
- 'persona' must not be empty
- 'persona' must be unique
- 'age_group' must be one of:
    - '18-20'
    - '21+'
- 'monthly_take_home_estimate' must be greater than zero  

  
For the seed table **'affordability_living_costs'**:
- 'cost_item' must not be empty
- 'cost_item' must be unique
- 'monthly_cost' must be greater than or equal to zero  

  
For the seed table **'affordability_transport'**:
- 'transport_option' must not be empty
- 'transport_option' must be unique
- 'monthly_transport_cost' must be greater than or equal to zero  
   

### 💡 Why this matters
  
  This is where DBT becomes more than just SQL.  
  
  It helps you say:  
  
  >   “These are the rules my data must follow.”  
  
  That is real-world analytics thinking.

---

# ❯❯❯❯ Part 3 — The Workshop Data Pipeline

We will now push our data through a real-world analytical workflow pipeline.

The pipeline steps are as follows:  

dbt debug / Verify dbt environment  
 ↓  
dbt seed / Load data  
 ↓  
inspect raw tables  
 ↓  
dbt run / Transform data  
 ↓  
dbt test / Validate outputs  
 ↓  
query final model  
 ↓  
inspect final model   ← qualitative understanding  
 ↓  
aggregate counts      ← quantitative insight  
 ↓  
Interpret results with charts/dashboard  
 ↓  
Present findings, business interpretation  

---

## 👩🏻‍💻 Step 3.1 — `debug` - Verify the DBT Project Configuration

Before we run any transformations, we should verify that our DBT project is configured correctly.

The `dbt debug` command checks:

- the Python environment
- the DBT installation
- the database connection
- the project configuration files
- the DuckDB database access

Think of this as a “system health check” before starting the analysis pipeline.

In real-world projects, engineers often run diagnostics before processing data.
This helps detect configuration problems early and avoids confusing downstream errors.


In [ ]:
debug_result = run_command(
  [
    venv_dbt,
    'debug',
    '--project-dir', dbt_project_dir,
    '--profiles-dir', dbt_project_dir
  ],
  cwd=repo_root, check=False
)

if debug_result.returncode == 0:
    print('✅ DBT debug passed')
else:
    raise RuntimeError('❌ DBT debug failed. Scroll up and read the output carefully.')
    

### ✅ Checkpoint — Understanding What Just Happened

If you saw:

✅ DBT debug passed

then DBT successfully verified that:

- your project files are valid
- your database connection works
- DuckDB can be accessed
- DBT can communicate with the database

You may also have noticed information such as:

- the database path
- the schema name
- the DBT version
- the Python version

This information is useful for troubleshooting and reproducibility.

At this stage:

- no data has been loaded yet
- no models have been created yet
- no transformations have been executed yet

We have simply confirmed that the analytical environment is ready.


❌ If this fails:
- check that `dbt_projects/p01_affordability/dbt_project.yml` exists
- check that `dbt_projects/p01_affordability/profiles.yml` exists
- re-run Step 6 and Step 7


## 🌱 Step 3.2 — `seed` - Load the Raw Data into the Database

We now move from project configuration into actual data engineering.

The `dbt seed` command takes CSV files from the `seeds/` folder and loads them into the DuckDB database as database tables.

This is an important transition point:

CSV files on disk 
↓
Database tables inside DuckDB 

Once the data is inside the database, we can:
- query it with SQL
- transform it with DBT models
- validate it with tests
- analyse it for business insights

In professional Data Analysis workflows, this step is often called:

- data ingestion
- data loading
- or data onboarding

In [ ]:
seed_result = run_command(
  [
    venv_dbt,
    'seed',
    '--project-dir', dbt_project_dir,
    '--profiles-dir', dbt_project_dir
  ],
  cwd=repo_root, check=False
)

if seed_result.returncode == 0:
    print('✅ Seeds loaded successfully')
else:
    raise RuntimeError('❌ DBT seed loading failed. Scroll up and read the output carefully.')
    

### ✅ Checkpoint — What Just Happened?

If you saw:

✅ Seeds loaded successfully

then DBT successfully loaded the CSV files from the `seeds/` folder into DuckDB.

At this stage, our raw datasets have become real database tables.

You may also have noticed messages such as:

- INSERT 12
- INSERT 4
- INSERT 5

These indicate how many rows were loaded into each table.

DBT has now created tables such as:

- affordability_rents
- affordability_personas
- affordability_living_costs
- affordability_transport

inside the database file:

`workshop.duckdb`

This is an important moment in the workflow:

raw files → structured database tables

We are now ready to inspect the database and begin transforming the data into analytical models.

---


## 🕵‍♀ Step 3.3 — Inspect the Raw Database Tables

At this stage, having loaded the CSV files into DuckDB, the data is no longer just a collection of files on disk.

The datasets now exist as real database tables inside DuckDB.

Before transforming the data, we will inspect these raw tables to understand:

- what information is available
- how the data is structured
- what issues or limitations may exist

This is an important part of real-world Data Analysis.

Analysts rarely begin by immediately building dashboards. 
They first explore and understand the raw data.


In [ ]:
# Connect to DuckDB

from pathlib import Path
import duckdb
import pandas as pd

db_path = dbt_project_dir / "workshop.duckdb"

print("Connecting to:", db_path)

con = duckdb.connect(str(db_path))

print("✅ Connected to DuckDB")


# Show Available Tables

tables = con.execute("""
SHOW TABLES
""").fetchdf()

print("Tables currently inside the database:")

display(tables)


We will now inspect the content of two "raw data" tables as they are in the database where they have been loaded.


In [ ]:
# Inspect One Raw Table

raw_rents = con.execute("""
SELECT *
FROM affordability_rents
LIMIT 10
""").fetchdf()

display(raw_rents)

In [ ]:
# Additional Inspection Cell

raw_personas = con.execute("""
SELECT *
FROM affordability_personas
LIMIT 10
""").fetchdf()

display(raw_personas)

con.close()

### ✅ Checkpoint — Understanding the Raw Data

You are now looking directly at the raw database tables created by `dbt seed`.

Notice that the data is:
- incomplete
- fragmented across multiple tables
- not yet easy to analyse directly

For example:
- rent data exists separately from income assumptions
- transport costs are isolated
- affordability has not yet been calculated

This is normal in real-world projects.

Raw data is often messy and disconnected.

The purpose of the next stage (`dbt run`) is to:
- combine datasets
- apply calculations
- introduce business logic
- produce analytical models that help answer real questions

In other words:

raw data
↓
transformation
↓
decision-ready insight


## 🎬 Step 3.4 — `run` - Transform the Raw Data into Analytical Models

So far, we have loaded raw datasets into DuckDB.

But raw data is rarely ready for business decision-making.

We now use the `dbt run` command to execute our DBT models.

These models:
- clean the raw data
- combine datasets together
- calculate useful metrics
- apply business logic
- create analytical outputs

This is the heart of the Data Analysis workflow.

DBT will now transform:

raw tables
↓
analytical models
↓
business-ready outputs

In professional environments, this stage is where data engineers and analysts turn disconnected raw information into structured knowledge that stakeholders can use.

**Note**:  DuckDB allows multiple readers, but write operations need an exclusive lock. dbt run needs to create or replace tables/views, so it needs that lock. Hence we close the notebook DuckDB connection before running DBT commands.


In [ ]:

# Preventive closing of possibly open DuckDB connection
try:
    con.close()
    print("✅ Closed DuckDB connection")
except NameError:
    print("No DuckDB connection variable named con was found")
except Exception as e:
    print("Could not close connection:", e)

    
# Execute the 'dbt run' command   
run_result = run_command(
    [
        venv_dbt,
        "run",
        "--project-dir", dbt_project_dir,
        "--profiles-dir", dbt_project_dir
    ],
    cwd=repo_root, check=False
)

if run_result.returncode == 0:
    print("✅ DBT run completed successfully")
else:
    raise RuntimeError("❌ DBT run failed. Scroll up and read the output carefully.")

### ✅ Checkpoint — What Did DBT Just Create?

If you saw:

✅ DBT run completed successfully

then DBT successfully executed your transformation pipeline.

This means DBT:
- read the raw seed tables
- executed the SQL transformation models
- created new analytical tables inside DuckDB

You may also have noticed messages such as:

- START model ...
- OK created model ...

These indicate that DBT executed SQL models and materialised them into database objects.

At this stage, the database now contains:

- raw input tables
- transformed staging models
- final business-ready models

This is one of the most important concepts in modern Data Analysis:

raw data is not the final product

The real value comes from transforming data into forms that help people understand problems and make decisions.

In the next steps, we will inspect these newly created models and analyse what they tell us about housing affordability in Brighton.


## 🧪 Step 3.5 — `test` - Validate the Quality of the Data

Building analytical models is not enough.

We also need to verify that the data is:
- complete
- valid
- internally consistent
- trustworthy

This is where DBT tests become important.

The `dbt test` command checks rules that we define about our data.

Examples include:
- values must not be missing
- numbers must be positive
- categories must belong to approved lists
- relationships between tables must be valid

In professional Data Analysis and Data Engineering projects, testing is essential because decisions based on poor-quality data can lead to serious mistakes.

Good analysts do not only produce results — they also verify that those results can be trusted.


In [ ]:
test_result = run_command(
    [
        venv_dbt,
        "test",
        "--project-dir", dbt_project_dir,
        "--profiles-dir", dbt_project_dir
    ],
    cwd=repo_root,
    check=False
)

if test_result.returncode == 0:
    print("✅ DBT tests passed — your data is consistent and trustworthy")
else:
    print("⚠️ Some tests failed — this is normal in real data projects. Scroll up to see what failed and why.")
    

### ✅ Checkpoint — What Did the Tests Verify?

Testing is the trust layer of your pipeline, if you saw:

#### ✅ DBT tests passed

then DBT successfully checked the quality and consistency of your data.

During this step, DBT executed validation rules against the database tables.

Examples of checks performed include:
- ensuring important values are not missing
- verifying that spending values are positive
- checking that categories belong to expected lists
- validating assumptions about the structure of the data

You may also have noticed messages such as:

- PASS
- FAIL
- WARN

These indicate whether each validation rule succeeded.

An important real-world lesson:

data projects are not only about creating outputs —
they are also about ensuring that those outputs are reliable.

Even experienced analysts encounter failing tests.
When that happens, the next step is investigation and debugging, not panic.

At this stage, we now have:
- loaded data
- transformed models
- validated analytical outputs

#### ❗ If the tests fail
  
  That means DBT found a data quality issue.  
  
  That is not a disaster — it is actually useful.  
  
  It means the tests are doing their job.

---

## 🔍 Step 3.6 — Inspect the Final Analytical Model

We have now:
- loaded raw data into DuckDB
- transformed the data using DBT
- validated the results with tests

The next step is one of the most important parts of Data Analysis:

examining the final analytical output.

We will now query the final DBT model:

`mart_affordability_scenarios`

This model represents the end result of the transformation pipeline.

It combines:
- rent information
- living costs
- transport costs
- income assumptions
- affordability calculations

into a single business-ready analytical table.

This is the type of output that analysts present to stakeholders and decision-makers and we can begin interpreting the results and to answer our original business question:

"Can young people still afford to live in Brighton?"

In [ ]:
query_code = textwrap.dedent(f'''
from pathlib import Path
import duckdb
import pandas as pd  # 👈 not strictly required, but clearer for learners

db_path = Path(r"{database_path}")
con = duckdb.connect(str(db_path))

query = con.execute("select * from mart_affordability_scenarios").fetchdf()

print(query.head(10).to_string(index=False))
print()
print("Rows:", len(query))
''')

# query_result = run_command([venv_python, '-c', query_code], cwd=repo_root)

query_result = run_command([
    venv_python, 
    '-c', query_code
], cwd=repo_root, check=False)


# if query_result.returncode != 0:
#    print("❌ Query failed")
    
if query_result.returncode == 0:
    print('✅ Query result successful')
else:
    print('❌ Query failed. Scroll up and read the output carefully.')
 


### ✅ Checkpoint — What Are We Looking At?

If you saw:

✅ Query result successful

then you successfully queried the final DBT analytical model from DuckDB, a database table that **you created**.

You are now viewing a business-ready dataset produced by the transformation pipeline and should be seeing rows from `mart_affordability_scenarios`.

Notice that this table is very different from the original raw CSV files.

The raw datasets were:
- fragmented
- incomplete
- difficult to interpret directly

The final model combines multiple datasets together and introduces:
- calculated affordability metrics
- classifications
- business logic
- stakeholder-oriented outputs

This is the real purpose of Data Analysis:

transforming disconnected raw data into meaningful information that supports decision-making.

At this stage, you are no longer simply “looking at data”.

You are beginning to answer the real-world question:

"Can young people still afford to live in Brighton?"

In the next steps, we will visualise and interpret these results to identify patterns, trends, and possible conclusions.

## 🔬 Step 3.7 — Explore the Final Analytical Model

We have now completed the transformation pipeline.

DBT has combined multiple raw datasets into a single analytical model designed to help answer our core question:

"Can young people still afford to live in Brighton?"

Before jumping into charts or conclusions, analysts first inspect the resulting dataset directly.

This is an important professional habit.

By looking at the final model, we can:
- understand the structure of the transformed data
- inspect calculated fields
- identify patterns
- verify that the outputs make sense

At this stage, we are no longer looking at raw operational data.

We are looking at a business-oriented analytical model created specifically to support decision-making.

In [ ]:
# Connect to DuckDB

from pathlib import Path
import duckdb
import pandas as pd

db_path = dbt_project_dir / "workshop.duckdb"

print("Connecting to:", db_path)

con = duckdb.connect(str(db_path))

print("✅ Connected to DuckDB")


# Show All Tables
# Connect to DuckDB

from pathlib import Path
import duckdb
import pandas as pd

db_path = dbt_project_dir / "workshop.duckdb"

print("Connecting to:", db_path)

con = duckdb.connect(str(db_path))

print("✅ Connected to DuckDB")


tables = con.execute("""
SHOW TABLES
""").fetchdf()

print("Tables currently inside the database:")

display(tables)


# Inspect the Final Business Model

final_model = con.execute("""
SELECT *
FROM mart_affordability_scenarios
ORDER BY affordability_status
""").fetchdf()

display(final_model)



### ✅ Checkpoint — What Changed Between Raw Data and Final Model?

Notice how different this table looks compared to the original raw datasets.

Earlier, the data was:
- spread across multiple tables
- difficult to interpret directly
- missing business context

The DBT transformation pipeline has now:
- combined related datasets
- calculated affordability measures
- added classifications and business logic
- produced a structured analytical output

This is one of the central goals of Data Analysis:

transforming disconnected raw information into meaningful, decision-ready insight.

You are now looking at the type of table that might be:
- queried by analysts
- visualised in dashboards
- used in reports
- presented to stakeholders
- discussed in meetings about housing policy or affordability concerns

---


## 📝 Step 3.8 — Summarise the Results

Inspecting individual rows helps us understand the structure of the data.

But decision-makers are usually interested in patterns and trends rather than single records.

We will now aggregate the data to answer questions such as:

- How many people fall into each affordability category?
- Which affordability situations are most common?
- What overall picture does the data suggest?

This is where Data Analysis begins moving from:
raw information
toward
interpretable evidence.


In [ ]:
# Count the Records

counts = con.execute("""
SELECT
    affordability_status,
    COUNT(*) AS number_of_people,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        1
    ) AS percentage
FROM mart_affordability_scenarios
GROUP BY affordability_status
ORDER BY number_of_people DESC
""").fetchdf()

display(counts)

con.close()

### ✅ Checkpoint — Turning Data into Insight

We have now moved beyond simply viewing data.

By grouping and counting records, we are beginning to identify meaningful patterns.

This type of aggregation helps transform large collections of rows into concise summaries that stakeholders can quickly understand.

For example, we can now begin discussing questions such as:

- Are most scenarios affordable or unaffordable?
- Which groups appear most financially pressured?
- What trends might concern local policymakers or community organisations?

This is a major transition point in the analytical workflow:

raw data
↓
transformed models
↓
aggregated summaries
↓
business insight

In the next steps, we will visualise these findings to make the patterns even clearer and easier to communicate.

---


# ❯❯❯❯ Part 4 — Analysis and Presentation

Once the analytical models have been created and validated, we can begin interpreting the results.

In this section, we:
- explore the transformed datasets
- identify patterns and trends
- summarise key findings
- create visualisations
- communicate conclusions

This is where technical data work becomes decision support.

The ultimate purpose of Data Analysis is not merely to process information, but to help people:
- understand problems
- evaluate alternatives
- make informed decisions

We will now use our analytical models to investigate the central workshop question:

"Can young people still afford to live in Brighton?"

# 1️⃣4️⃣ Analyse the Results

Now we use Python to explore the decision-ready table.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect(str(database_path))
affordability = con.execute("select * from mart_affordability_scenarios").fetchdf()
affordability.head()

## ✅ Checkpoint

You should see a dataframe.

# 1️⃣5️⃣ Summary: How Many Scenarios Are Affordable?

In [ ]:
status_summary = (
    affordability
    .groupby("affordability_status")
    .size()
    .reset_index(name="scenario_count")
    .sort_values("scenario_count", ascending=False)
)
status_summary

## ✅ Checkpoint

You should see counts for each affordability status.

# 1️⃣6️⃣ Chart: Affordability Status Counts

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(status_summary["affordability_status"], status_summary["scenario_count"])
plt.title("How many Brighton living scenarios are affordable?")
plt.xlabel("Affordability status")
plt.ylabel("Number of scenarios")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## ✅ Checkpoint

You should see a bar chart.

# 1️⃣7️⃣ Focus on Shared Rooms

For many young people, shared housing is the first realistic option.

In [ ]:
shared = affordability[affordability["rent_option"] == "shared_room"].copy()
shared_summary = (
    shared
    .groupby(["persona", "affordability_status"])
    .size()
    .reset_index(name="scenario_count")
)
shared_summary

## ✅ Checkpoint

You should see shared-room affordability counts by persona.

# 1️⃣8️⃣ Chart: Average Leftover by Persona

In [ ]:
leftover_by_persona = (
    affordability
    .groupby("persona")["monthly_leftover"]
    .mean()
    .sort_values()
    .reset_index()
)
plt.figure(figsize=(10, 5))
plt.barh(leftover_by_persona["persona"], leftover_by_persona["monthly_leftover"])
plt.title("Average monthly leftover by persona")
plt.xlabel("Average monthly leftover (£)")
plt.ylabel("Persona")
plt.tight_layout()
plt.show()
leftover_by_persona

## ✅ Checkpoint

Ask: which young person is most financially squeezed?

# 1️⃣9️⃣ Chart: Rent Burden by Rent Option

Rent burden means `rent ÷ income × 100`.

In [ ]:
rent_burden = (
    affordability
    .groupby(["persona", "rent_option"])["rent_burden_pct"]
    .mean()
    .reset_index()
)
pivot_burden = rent_burden.pivot(index="persona", columns="rent_option", values="rent_burden_pct")
pivot_burden.plot(kind="bar", figsize=(10, 5))
plt.title("Average rent burden by persona and rent option")
plt.xlabel("Persona")
plt.ylabel("Rent burden (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
pivot_burden

## ✅ Checkpoint

You should see that one-bed renting is much harder than shared renting.

# 2️⃣0️⃣ Decision Table: Best Options for Each Persona

In [ ]:
best_options = (
    affordability
    .sort_values(["persona", "monthly_leftover"], ascending=[True, False])
    .groupby("persona")
    .head(5)
    [["persona", "area", "rent_option", "transport_option", "monthly_take_home_estimate", "monthly_rent", "monthly_transport_cost", "monthly_leftover", "rent_burden_pct", "affordability_status"]]
)
best_options

## ✅ Checkpoint

You should see the top five options for each persona.

# 2️⃣1️⃣ Simple Dashboard: Could I Afford Brighton?

Change the values below and run the cell.

In [ ]:
selected_persona = "19-year-old retail worker"
selected_area = "Moulsecoomb"
selected_rent_option = "shared_room"
selected_transport_option = "bus_commuter"

choice = affordability[
    (affordability["persona"] == selected_persona)
    & (affordability["area"] == selected_area)
    & (affordability["rent_option"] == selected_rent_option)
    & (affordability["transport_option"] == selected_transport_option)
]

if choice.empty:
    print("No matching scenario found. Check your spelling.")
else:
    row = choice.iloc[0]
    print("🏠 Brighton affordability checker")
    print("--------------------------------")
    print("Persona:", row["persona"])
    print("Area:", row["area"])
    print("Rent option:", row["rent_option"])
    print("Transport:", row["transport_option"])
    print()
    print("Monthly take-home estimate: £", round(row["monthly_take_home_estimate"], 2))
    print("Monthly rent: £", round(row["monthly_rent"], 2))
    print("Monthly transport: £", round(row["monthly_transport_cost"], 2))
    print("Monthly base living costs: £", round(row["monthly_base_living_cost"], 2))
    print("Monthly leftover: £", round(row["monthly_leftover"], 2))
    print("Rent burden:", round(row["rent_burden_pct"], 1), "%")
    print()
    print("Status:", row["affordability_status"])

## ✅ Checkpoint

Try changing the selected values to test different scenarios.

# 2️⃣2️⃣ Optional: Create a Streamlit App

This creates a small app file that can be run later with:

```bash
streamlit run 03_project/brighton_affordability_app.py
```

In [ ]:
app_file = repo_root / "03_project" / "brighton_affordability_app.py"
app_template = '''
import duckdb
import streamlit as st

st.set_page_config(page_title="Brighton Affordability Checker", layout="centered")
st.title("🏠 Could You Afford Brighton?")
st.write("A simple workshop app using DBT output data.")

db_path = r"__DB_PATH__"
con = duckdb.connect(db_path)
df = con.execute("select * from mart_affordability_scenarios").fetchdf()

persona = st.selectbox("Choose a persona", sorted(df["persona"].unique()))
area = st.selectbox("Choose an area", sorted(df["area"].unique()))
rent_option = st.selectbox("Choose rent option", sorted(df["rent_option"].unique()))
transport_option = st.selectbox("Choose transport option", sorted(df["transport_option"].unique()))

selected = df[
    (df["persona"] == persona)
    & (df["area"] == area)
    & (df["rent_option"] == rent_option)
    & (df["transport_option"] == transport_option)
]

if not selected.empty:
    row = selected.iloc[0]
    st.metric("Monthly leftover", f"£{row['monthly_leftover']:.0f}")
    st.metric("Rent burden", f"{row['rent_burden_pct']:.1f}%")
    st.subheader(f"Status: {row['affordability_status']}")
    st.write("### Scenario details")
    st.dataframe(selected)

st.write("---")
st.write("This is a teaching app. Figures are simplified workshop estimates based on public-source benchmarks.")
'''
app_file.write_text(app_template.replace("__DB_PATH__", str(database_path)), encoding="utf-8")
print("✅ Streamlit app created:")
print(app_file)

## ✅ Checkpoint

You should see `03_project/brighton_affordability_app.py` created.

In [ ]:
# Close DuckDB connection if it is open

try:
    con.close()
except:
    pass

# Stop old Streamlit process if it exists
try:
    streamlit_process.terminate()
    print("Stopped old Streamlit process")
except:
    pass



import subprocess
from pathlib import Path

app_path = repo_root / "03_project" / "brighton_affordability_app.py"

streamlit_process = subprocess.Popen(
    [
        str(venv_python),
        "-m",
        "streamlit",
        "run",
        str(app_path),
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    cwd=str(repo_root),
)

print("✅ Streamlit is starting")
print("Open this in your Chromebook browser:")
print("http://localhost:8501")
print()
print("If that does not work, also try:")
print("http://127.0.0.1:8501")


In [ ]:
streamlit_process.terminate()

# 2️⃣3️⃣ Write Your Analyst Findings

Now move from code to communication.

In [ ]:
not_affordable_count = int((affordability["affordability_status"] == "Not affordable").sum())
total_count = len(affordability)
not_affordable_pct = round(not_affordable_count * 100 / total_count, 1)
most_stretched = leftover_by_persona.iloc[0]["persona"]
lowest_rent_area = rents.groupby("area")["monthly_rent"].mean().sort_values().index[0]

print("Automatic findings:")
print(f"1. {not_affordable_pct}% of scenarios are classified as not affordable.")
print(f"2. The most financially stretched persona is: {most_stretched}.")
print(f"3. The area with the lowest average rent in this teaching dataset is: {lowest_rent_area}.")

## ✅ Checkpoint

You should see three automatically generated findings.

# 2️⃣4️⃣ Frame Alternatives for a Stakeholder

A stakeholder does not just need facts. They need choices.

| Option | What it means | Who it helps |
|---|---|---|
| Shared housing support | Help young people find safe shared housing | Apprentices and lower-paid workers |
| Transport support | Reduce commuting cost pressure | People living farther from central Brighton |
| Local wage partnerships | Encourage employers to pay above minimum wage | Young full-time workers |
| Affordable starter housing | Increase access to lower-cost rentals | Young people trying to live independently |
| Career pathway support | Help young residents move into higher-paid digital roles | Students and early-career workers |

# 2️⃣5️⃣ Create a Stakeholder Summary

This cell writes a short markdown report you can keep.

In [ ]:
summary_file = output_dir / "brighton_affordability_summary.md"
summary_text = f"""# Brighton Affordability Analysis Summary

## Question

Can young people still afford to live in Brighton?

## Data Used

This analysis used prepared workshop datasets based on public-source benchmarks for:

- Brighton rents
- young worker income scenarios
- essential living costs
- transport cost scenarios

## Key Findings

1. {not_affordable_pct}% of modelled scenarios were classified as **not affordable**.
2. The most financially stretched persona was: **{most_stretched}**.
3. The area with the lowest average rent in this teaching dataset was: **{lowest_rent_area}**.

## Decision Alternatives

Stakeholders could consider:

1. Support safe shared housing options for young people.
2. Improve transport affordability for young workers.
3. Work with local employers on better early-career pay.
4. Expand affordable starter housing options.
5. Help young people access higher-paid digital and analytical career pathways.

## Important Caveat

This is a workshop model. The data is simplified for teaching and should be replaced with a fuller official dataset before making real policy decisions.
"""
summary_file.write_text(summary_text, encoding="utf-8")
print("✅ Summary report written:")
print(summary_file)
print()
print(summary_text)

## ✅ Checkpoint

You should now have:

`03_project/output/brighton_affordability_summary.md`

# 🎉 Final Reflection

You have completed a full data analysis process:

```text
Question → Data → DBT Transformation → Tests → Analysis → Visuals → Decision Alternatives
```

## CV-style description

> Built a Brighton housing affordability analysis using Python, DuckDB and DBT. Modelled young-person income scenarios, rent options and living costs to identify affordability pressures and decision alternatives for local stakeholders.

## Future extensions

- use live official datasets
- include more Brighton neighbourhoods
- add council tax estimates
- add salary data by occupation
- compare Brighton with nearby towns
- host the Streamlit app as a portfolio project